# Convert PyTorch checkpoints to HDF5

This refreshes the GAN generator checkpoint in `models/part_a/`. The three VAE HDF5 exports already match their PyTorch checkpoints exactly, so this rerun leaves them untouched. The weights remain a PyTorch state dictionary; changing the container to `.h5` does not turn it into a Keras model.

In [ ]:
# Run this once if h5py is not installed in the current notebook kernel.
%pip install -q h5py

In [1]:
from pathlib import Path

import h5py
import torch

checkpoint_files = [
    Path("notebooks/part_a/best_gan.pt"),
]

output_dir = Path("models/part_a")
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
def convert_pt_to_h5(pt_path, output_dir):
    """Copy one PyTorch state dictionary into an HDF5 file."""
    state_dict = torch.load(pt_path, map_location="cpu", weights_only=True)
    h5_path = output_dir / f"{pt_path.stem}.h5"

    with h5py.File(h5_path, "w") as h5_file:
        h5_file.attrs["framework"] = "pytorch"
        h5_file.attrs["format"] = "state_dict"
        h5_file.attrs["source_file"] = pt_path.name
        weights = h5_file.create_group("state_dict")

        for name, tensor in state_dict.items():
            weights.create_dataset(name, data=tensor.detach().cpu().numpy())

    return h5_path


for checkpoint in checkpoint_files:
    if not checkpoint.exists():
        print(f"Skipped missing file: {checkpoint}")
        continue

    converted = convert_pt_to_h5(checkpoint, output_dir)
    with h5py.File(converted, "r") as h5_file:
        tensor_count = len(h5_file["state_dict"])
    print(f"Created {converted} ({tensor_count} tensors)")

Created models\part_a\best_gan.h5 (23 tensors)
